In [0]:
customers_bronze = spark.table(
    "workspace.banking_bronze.customers"
)

print("Bronze records:", customers_bronze.count())

In [0]:
customers_bronze = spark.table(
    "workspace.banking_bronze.customers"
)

In [0]:
print("Bronze records:", customers_bronze.count())

In [0]:
customers_silver = customers_bronze.dropDuplicates(["customer_id"])

In [0]:
print("Silver records:", customers_silver.count())

In [0]:
customers_silver = customers_bronze.dropDuplicates(["customer_id"])

print("Bronze records:", customers_bronze.count())
print("Silver records:", customers_silver.count())

In [0]:
from pyspark.sql.functions import col, sum

In [0]:
customers_silver.select(
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("customer_name").isNull().cast("int")).alias("name_nulls"),
    sum(col("email").isNull().cast("int")).alias("email_nulls"),
    sum(col("age").isNull().cast("int")).alias("age_nulls"),
    sum(col("city").isNull().cast("int")).alias("city_nulls"),
    sum(col("state").isNull().cast("int")).alias("state_nulls"),
    sum(col("registration_date").isNull().cast("int")).alias("date_nulls")
).show()

In [0]:
from pyspark.sql.functions import coalesce, lit
customers_silver = customers_silver.withColumn(
    "email",
    coalesce(col("email"), lit("Unknown"))
)

In [0]:
customers_silver = customers_silver.withColumn(
    "city",
    coalesce(col("city"), lit("Unknown"))
)

In [0]:
invalid_age_count = customers_silver.filter(
    (col("age") < 18) | (col("age") > 100)
).count()

print("Invalid age records:", invalid_age_count)

In [0]:
customers_silver.show(10)

In [0]:
customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.banking_silver.customers")

In [0]:
spark.table("workspace.banking_silver.customers").count()

In [0]:
accounts_bronze = spark.table(
    "workspace.banking_bronze.accounts"
)

print("Bronze account records:", accounts_bronze.count())

In [0]:
accounts_bronze.show(5)

In [0]:
print("Total records:", accounts_bronze.count())

print(
    "Unique account IDs:",
    accounts_bronze.select("account_id").distinct().count()
)

In [0]:
accounts_silver = accounts_bronze.dropDuplicates(["account_id"])

print("Before:", accounts_bronze.count())
print("After :", accounts_silver.count())

In [0]:
from pyspark.sql.functions import col, sum

accounts_silver.select(
    sum(col("account_id").isNull().cast("int")).alias("account_id_nulls"),
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("branch_id").isNull().cast("int")).alias("branch_id_nulls"),
    sum(col("account_type").isNull().cast("int")).alias("account_type_nulls"),
    sum(col("opening_date").isNull().cast("int")).alias("opening_date_nulls"),
    sum(col("initial_balance").isNull().cast("int")).alias("balance_nulls")
).show()

In [0]:
print("Total records:", accounts_bronze.count())

print(
    "Unique account IDs:",
    accounts_bronze.select("account_id").distinct().count()
)

In [0]:
accounts_silver = accounts_bronze.dropDuplicates(["account_id"])

print("Before:", accounts_bronze.count())
print("After :", accounts_silver.count())

In [0]:
from pyspark.sql.functions import col, sum

accounts_silver.select(
    sum(col("account_id").isNull().cast("int")).alias("account_id_nulls"),
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("branch_id").isNull().cast("int")).alias("branch_id_nulls"),
    sum(col("account_type").isNull().cast("int")).alias("account_type_nulls"),
    sum(col("opening_date").isNull().cast("int")).alias("opening_date_nulls"),
    sum(col("initial_balance").isNull().cast("int")).alias("balance_nulls")
).show()

In [0]:
accounts_bronze = spark.table(
    "workspace.banking_bronze.accounts"
)

print("Total records:", accounts_bronze.count())

In [0]:
accounts_silver = accounts_bronze.dropDuplicates(
    ["account_id"]
)

print("Before:", accounts_bronze.count())
print("After:", accounts_silver.count())

In [0]:
from pyspark.sql.functions import col, sum

accounts_silver.select(
    sum(col("account_id").isNull().cast("int")).alias("account_id_nulls"),
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("branch_id").isNull().cast("int")).alias("branch_id_nulls"),
    sum(col("account_type").isNull().cast("int")).alias("account_type_nulls"),
    sum(col("opening_date").isNull().cast("int")).alias("opening_date_nulls"),
    sum(col("initial_balance").isNull().cast("int")).alias("balance_nulls")
).show()

In [0]:
accounts_silver.groupBy("account_type").count().show()

In [0]:
negative_balance_count = accounts_silver.filter(
    col("initial_balance") < 0
).count()

print("Negative balance records:", negative_balance_count)

In [0]:
accounts_silver.select(
    "opening_date"
).describe().show()

In [0]:
accounts_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.banking_silver.accounts"
    )

In [0]:
spark.table(
    "workspace.banking_silver.accounts"
).count()

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    lit,
    row_number
)

from pyspark.sql.window import Window

In [0]:
transactions_bronze = spark.table("banking_bronze.transactions")

print("Bronze Transactions Records:", transactions_bronze.count())

transactions_bronze.show(10)
transactions_bronze.printSchema()

In [0]:
from pyspark.sql.functions import sum as spark_sum

transactions_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_bronze.columns
]).show()

In [0]:
total_records = transactions_bronze.count()

unique_transactions = transactions_bronze.select(
    "transaction_id"
).distinct().count()

print("Total records:", total_records)
print("Unique transaction IDs:", unique_transactions)
print("Duplicate records:", total_records - unique_transactions)

In [0]:
transactions_bronze.groupBy("transaction_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_silver = transactions_bronze.dropDuplicates()

print("Records after duplicate removal:", transactions_silver.count())

In [0]:
window_spec = Window.partitionBy("transaction_id").orderBy(
    col("transaction_date").desc()
)

transactions_silver = (
    transactions_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Records after transaction ID deduplication:",
      transactions_silver.count())

In [0]:
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("transaction_type", trim(col("transaction_type")))
)

In [0]:
transactions_silver = transactions_silver.withColumn(
    "transaction_type",
    upper(trim(col("transaction_type")))
)

transactions_silver.groupBy("transaction_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_silver.select("transaction_date").show(20, False)

In [0]:
transactions_silver = transactions_silver.withColumn(
    "transaction_date",
    to_date(col("transaction_date"))
)

In [0]:
transactions_silver = transactions_silver.withColumn(
    "amount",
    col("amount").cast("double")
)

In [0]:
transactions_silver = transactions_silver.withColumn(
    "balance",
    col("balance").cast("double")
)

In [0]:
transactions_silver = transactions_silver.filter(
    col("amount") > 0
)

In [0]:
transactions_silver = transactions_silver.filter(
    col("transaction_id").isNotNull()
)

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

transactions_bronze = spark.table("banking_bronze.transactions")

print("Columns:")
print(transactions_bronze.columns)

print("Schema:")
transactions_bronze.printSchema()

print("Total records:", transactions_bronze.count())

In [0]:
transactions_silver = transactions_bronze.dropDuplicates()

print("Records after duplicate removal:",
      transactions_silver.count())

In [0]:
window_spec = Window.partitionBy(
    "transaction_id"
).orderBy(
    col("transaction_date").desc()
)

transactions_silver = (
    transactions_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Records after transaction ID deduplication:",
      transactions_silver.count())

In [0]:
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn(
        "transaction_type",
        upper(trim(col("transaction_type")))
    )
)

In [0]:
transactions_silver = (
    transactions_silver
    .withColumn(
        "transaction_date",
        to_date(col("transaction_date"))
    )
    .withColumn(
        "amount",
        col("amount").cast("double")
    )
)

In [0]:
transactions_silver.printSchema()

transactions_silver.show(10, False)

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

transactions_bronze = spark.table("banking_bronze.transactions")

print("Columns:", transactions_bronze.columns)
print("Total records:", transactions_bronze.count())

transactions_bronze.printSchema()

In [0]:
total_records = transactions_bronze.count()

unique_transaction_ids = transactions_bronze \
    .select("transaction_id") \
    .distinct() \
    .count()

print("Total records:", total_records)
print("Unique transaction IDs:", unique_transaction_ids)
print("Duplicate transaction IDs:",
      total_records - unique_transaction_ids)

In [0]:
from pyspark.sql.functions import sum as spark_sum

transactions_bronze.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_bronze.columns
]).show()

In [0]:
transactions_bronze.groupBy("transaction_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_bronze.groupBy("payment_method") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_bronze.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_bronze.filter(
    col("amount") <= 0
).show(20, False)

In [0]:
transactions_bronze.groupBy("transaction_type") \
    .agg(
        spark_sum("amount").alias("total_amount"),
        spark_sum(
            (col("amount") < 0).cast("int")
        ).alias("negative_count")
    ) \
    .orderBy("transaction_type") \
    .show()

In [0]:
transactions_bronze.groupBy(
    "transaction_type"
).agg(
    spark_sum(
        (col("amount") > 0).cast("int")
    ).alias("positive_count"),
    
    spark_sum(
        (col("amount") < 0).cast("int")
    ).alias("negative_count"),
    
    spark_sum(
        (col("amount") == 0).cast("int")
    ).alias("zero_count")
).show()

In [0]:
transactions_bronze.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_bronze.groupBy("payment_method") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
transactions_bronze.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_bronze.columns
]).show()

In [0]:
transactions_bronze.filter(
    col("payment_method").isNull() |
    col("merchant_id").isNull()
).show(30, False)

In [0]:
transactions_bronze.filter(
    col("payment_method").isNull()
).select(
    "transaction_id",
    "transaction_type",
    "amount",
    "payment_method",
    "merchant_id",
    "status"
).show(20, False)

In [0]:
transactions_bronze.filter(
    col("merchant_id").isNull()
).select(
    "transaction_id",
    "transaction_type",
    "amount",
    "payment_method",
    "merchant_id",
    "status"
).show(20, False)

In [0]:
transactions_silver = transactions_bronze


In [0]:
transactions_silver = transactions_silver.dropDuplicates()

print(
    "Records after removing exact duplicates:",
    transactions_silver.count()
)

In [0]:
window_spec = Window.partitionBy(
    "transaction_id"
).orderBy(
    col("transaction_date").desc()
)

transactions_silver = (
    transactions_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print(
    "Records after transaction ID deduplication:",
    transactions_silver.count()
)

In [0]:
transactions_silver = (
    transactions_silver
    .withColumn(
        "transaction_id",
        trim(col("transaction_id"))
    )
    .withColumn(
        "account_id",
        trim(col("account_id"))
    )
    .withColumn(
        "transaction_type",
        upper(trim(col("transaction_type")))
    )
    .withColumn(
        "payment_method",
        upper(trim(col("payment_method")))
    )
    .withColumn(
        "status",
        upper(trim(col("status")))
    )
)

In [0]:
from pyspark.sql.functions import coalesce

transactions_silver = transactions_silver.withColumn(
    "merchant_id",
    coalesce(
        trim(col("merchant_id")),
        lit("UNKNOWN")
    )
)

In [0]:
transactions_silver.groupBy(
    "transaction_type"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
transactions_silver.groupBy(
    "payment_method"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
transactions_silver.groupBy(
    "status"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
transactions_silver.groupBy(
    "transaction_type"
).agg(
    spark_sum(
        (col("amount") > 0).cast("int")
    ).alias("positive_count"),
    
    spark_sum(
        (col("amount") < 0).cast("int")
    ).alias("negative_count"),
    
    spark_sum(
        (col("amount") == 0).cast("int")
    ).alias("zero_count")
).show()

In [0]:
negative_count = transactions_silver.filter(
    col("amount") < 0
).count()

print("Negative amount records:", negative_count)

transactions_silver = transactions_silver.filter(
    col("amount") > 0
)

print(
    "Records after removing negative amounts:",
    transactions_silver.count()
)

In [0]:
transactions_silver.select(
    "amount"
).summary().show()

In [0]:
transactions_silver.select(
    "transaction_date"
).summary().show()

In [0]:
print(
    "Minimum transaction date:",
    transactions_silver.select("transaction_date")
    .agg({"transaction_date": "min"})
    .collect()[0][0]
)

print(
    "Maximum transaction date:",
    transactions_silver.select("transaction_date")
    .agg({"transaction_date": "max"})
    .collect()[0][0]
)

In [0]:
print(
    "Total records:",
    transactions_silver.count()
)

print(
    "Unique transaction IDs:",
    transactions_silver
    .select("transaction_id")
    .distinct()
    .count()
)

In [0]:
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
transactions_silver.orderBy(
    "transaction_date"
).show(20, False)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lit,
    coalesce,
    sum as spark_sum
)

In [0]:
transactions_bronze = spark.table("banking_bronze.transactions")

transactions_silver = transactions_bronze

print("Bronze records:", transactions_bronze.count())

In [0]:
transactions_silver = transactions_silver.dropDuplicates()

print(
    "After exact duplicate removal:",
    transactions_silver.count()
)

In [0]:
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("status", upper(trim(col("status"))))
)

In [0]:
transactions_silver = transactions_silver.withColumn(
    "merchant_id",
    coalesce(
        trim(col("merchant_id")),
        lit("UNKNOWN")
    )
)

In [0]:
transactions_silver.filter(
    col("merchant_id") == "UNKNOWN"
).count()

In [0]:
print(
    "Negative amount records:",
    transactions_silver.filter(
        col("amount") < 0
    ).count()
)